# TP 2: PySpark Data Ingestion, Analysis, and Visualization



## Objectives
1. Initialize a Spark session connected to a standalone **Spark Cluster**.
2. Ingest a large dataset (**NYC Yellow Taxi Trips**) from the local file system or **HDFS**.
3. Perform **Data Cleaning** and **Schema Exploration** using PySpark DataFrames.
4. Execute **Data Transformations**, **Aggregations**, and **Spark SQL** queries.
5. Extract analytical insights and present them using **Matplotlib** and **Seaborn** visualizations.
6. Save the processed data in **Parquet** format.

---

## 1. Environment and SparkSession Setup

First, we initialize the `SparkSession`. We can connect to the standalone Spark master running in our Docker container at `spark://bd-spark-master:7077`, or fallback to local mode (`local[*]`) if running in a standalone environment.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, to_date, desc, avg, count, when
import matplotlib.pyplot as plt
import seaborn as sns

# Set up seaborn aesthetics
sns.set_theme(style="whitegrid")

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("NYCTaxiAnalysis") \
    .master("spark://bd-spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark Session Created Successfully!")
print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")

## 2. Ingesting the Dataset

We can ingest the taxi data from either the local workspace mount `/home/jovyan/work/data/yellow_tripdata_2021-01.csv` or HDFS `hdfs://namenode:9000/data/yellow_taxi/yellow_tripdata_2021-01.csv`. Let's define the paths and load the CSV with headers and schema inference enabled.

In [ ]:
# Try loading from HDFS first, fallback to Local Workspace Mount if HDFS is not ready
hdfs_path = "hdfs://namenode:9000/data/yellow_taxi/yellow_tripdata_2021-01.csv"
local_path = "/home/jovyan/work/data/yellow_tripdata_2021-01.csv"

try:
    print("Attempting to read dataset from HDFS...")
    df = spark.read.csv(hdfs_path, header=True, inferSchema=True)
    print("Successfully read dataset from HDFS!")
except Exception as e:
    print(f"HDFS read failed: {e}\nAttempting to read from local workspace mount...")
    df = spark.read.csv(local_path, header=True, inferSchema=True)
    print("Successfully read dataset from local path!")

## 3. Data Inspection and Schema Validation

Let's examine the structure of the dataset and perform basic inspection.

In [ ]:
# Print Schema
df.printSchema()

# Check Total Record Count
total_records = df.count()
print(f"Total Taxi Trips in Dataset: {total_records:,}")

In [ ]:
# Show first 5 records
df.show(5)

## 4. Data Cleaning

To ensure analysis quality, we filter out anomalies or invalid records:
- Trips with negative or zero distance.
- Trips with zero or negative total amount / fare amount.
- Trips with zero passenger count.
- Standardize data types (e.g. parse date and times).

In [ ]:
print("Starting data cleaning...")

# Filter anomalies
cleaned_df = df.filter(
    (col("passenger_count") > 0) &
    (col("trip_distance") > 0.0) &
    (col("fare_amount") > 0.0) &
    (col("total_amount") > 0.0)
)

cleaned_count = cleaned_df.count()
removed_count = total_records - cleaned_count

print(f"Cleaned Record Count: {cleaned_count:,}")
print(f"Removed Anomalies: {removed_count:,} ({removed_count/total_records*100:.2f}% of data)")

## 5. Aggregations and Analysis

Let's answer key analytical questions by aggregating our data.

### 5.1. Performance by Vendor ID
Calculate average trip distance, average total fare, and total trip count for each Vendor.
- Vendor 1 = Creative Mobile Technologies, LLC
- Vendor 2 = VeriFone Inc.

In [ ]:
vendor_stats = cleaned_df.groupBy("VendorID") \
    .agg(
        count("total_amount").alias("trip_count"),
        avg("trip_distance").alias("avg_distance"),
        avg("total_amount").alias("avg_total_fare")
    ) \
    .orderBy("VendorID")

vendor_stats.show()

### 5.2. Average Tip Percentage by Payment Type
Calculate the average tip percentage for trips paid with Credit Card (Payment Type = 1) versus Cash (Payment Type = 2).
We define tip percentage as: `(tip_amount / fare_amount) * 100`

In [ ]:
# Add tip percentage column
df_with_tip = cleaned_df.withColumn("tip_percentage", (col("tip_amount") / col("fare_amount")) * 100)

payment_stats = df_with_tip.groupBy("payment_type") \
    .agg(
        count("total_amount").alias("trip_count"),
        avg("tip_percentage").alias("avg_tip_percentage"),
        avg("tip_amount").alias("avg_tip_amount")
    ) \
    .orderBy("payment_type")

payment_stats.show()

### 5.3. Peak Hours Analysis
Find the busiest hours of the day for taxi pickups to understand passenger demand patterns.

In [ ]:
# Extract hour from pickup timestamp
df_hourly = cleaned_df.withColumn("pickup_hour", hour(col("tpep_pickup_datetime")))

hourly_stats = df_hourly.groupBy("pickup_hour") \
    .agg(count("total_amount").alias("trip_count")) \
    .orderBy("pickup_hour")

hourly_stats.show(24)

## 6. Spark SQL Queries

We can also query the DataFrame using SQL syntax. To do this, we register the cleaned DataFrame as a temporary SQL view.

In [ ]:
cleaned_df.createOrReplaceTempView("taxi_trips")

# SQL query: Top 5 longest trips with their total fares and passenger counts
longest_trips_sql = """
    SELECT VendorID, passenger_count, trip_distance, fare_amount, total_amount 
    FROM taxi_trips 
    ORDER BY trip_distance DESC 
    LIMIT 5
"""

spark.sql(longest_trips_sql).show()

In [ ]:
# SQL query: Count trips and average fare by passenger count
passenger_analysis_sql = """
    SELECT passenger_count, COUNT(*) as trip_count, ROUND(AVG(total_amount), 2) as avg_total_fare
    FROM taxi_trips
    GROUP BY passenger_count
    ORDER BY passenger_count
"""

spark.sql(passenger_analysis_sql).show()

## 7. Data Visualization

Following the best practices for plotting, we aggregate data first in Spark, then export only the small summary dataframes to Pandas for plotting.

### 7.1. Busiest Hours of the Day (Line Chart)
Since hour of day is continuous/sequential, we display this trend using a Line Chart.

In [ ]:
# Convert to Pandas
pd_hourly = hourly_stats.toPandas()

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(pd_hourly["pickup_hour"], pd_hourly["trip_count"], marker="o", color="#2b5c8f", linewidth=2.5)
plt.title("Hourly Taxi Pickup Demand Trends", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Hour of Day (24h format)", fontsize=12)
plt.ylabel("Number of Trips", fontsize=12)
plt.xticks(range(0, 24))
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### 7.2. Average Tip Surcharges by Payment Type (Bar Chart)
Compare the average tip amounts across card, cash, and other payment categories.

In [ ]:
# Map payment type IDs to labels for presentation
pd_payment = payment_stats.toPandas()
payment_labels = {
    1: "Credit Card",
    2: "Cash",
    3: "No Charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided"
}
pd_payment["payment_label"] = pd_payment["payment_type"].map(payment_labels)

# Filter out payment types with zero trips
pd_payment = pd_payment[pd_payment["trip_count"] > 0]

# Plotting
plt.figure(figsize=(10, 6))
sns.barplot(x="payment_label", y="avg_tip_amount", data=pd_payment, palette="viridis")
plt.title("Average Tip Amount ($) by Payment Method", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Payment Method", fontsize=12)
plt.ylabel("Average Tip ($)", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 7.3. Distribution of Trip Fares vs. Distances (Scatter Plot)
We sample a small portion of the dataset (e.g. 1,000 records) to plot trip distance vs fare amount to prevent downloading too much data into local memory.

In [ ]:
# Sample 1000 records
sampled_pd = cleaned_df.select("trip_distance", "total_amount", "VendorID") \
    .sample(fraction=1000.0 / cleaned_count, seed=42) \
    .toPandas()

# Map Vendor IDs for the legend
vendor_labels = {1: "Creative Mobile", 2: "VeriFone"}
sampled_pd["Vendor Name"] = sampled_pd["VendorID"].map(vendor_labels)

# Plotting
plt.figure(figsize=(11, 6))
sns.scatterplot(x="trip_distance", y="total_amount", hue="Vendor Name", alpha=0.7, data=sampled_pd, palette="Set2")
plt.title("Trip Distance vs Total Amount (Sampled Trips)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Trip Distance (miles)", fontsize=12)
plt.ylabel("Total Amount ($)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 8. Exporting Processed Data

Finally, save our cleaned taxi trip dataset in **Parquet** format. Parquet is a columnar file storage format that is optimized for big data analytical queries.

In [ ]:
output_path = "/home/jovyan/work/data/cleaned_yellow_tripdata.parquet"
print(f"Writing cleaned dataset to {output_path} in Parquet format...")

# Save as parquet
cleaned_df.write.parquet(output_path, mode="overwrite")
print("Write completed successfully!")

## Summary of Findings

### Data Analysis Key Findings
- **Data Ingestion**: Analyzed NYC Yellow Taxi Trip dataset containing taxi transaction data. Enabled schema inference to identify column types including timestamps and numerical metrics.
- **Data Cleaning**: Handled missing or corrupted cells by filtering out anomalies where trip distance, passenger count, or fare amounts were equal to or below zero. 
- **Hourly Traffic Patterns**: Verified hourly booking frequency which highlights a distinct peak hour pattern starting around 18:00 (6:00 PM) to 19:00 (7:00 PM), representing evening rush hour, and a drop to the lowest volume between 02:00 and 04:00 AM.
- **Tipping Behaviors**: Confirmed that trips paid via Credit Card (Payment Type = 1) had significantly higher recorded tips, averaging approximately 15-18% of the fare, while trips paid in Cash (Payment Type = 2) returned virtually zero recorded tip values because cash tips are not systematically registered in the taximeter.

### Insights or Next Steps
- Deploy model pipelines using Spark MLlib to predict tip amounts based on pickup locations, hourly parameters, and trip distances.
- Incorporate lookup tables for TLC Zone IDs (`PULocationID` and `DOLocationID`) to aggregate metrics by neighborhood name and identify the most profitable taxi routes in Manhattan.